# Caso E · 03 Features meteorológicos para Caso B

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir features meteorológicas reusables por el modelo de consumo del Caso B.


## 2. Qué se aprende

- Lag/lead temporal de variables exógenas.
- Dewpoint, sensación térmica.
- Diurnal/seasonal encoding.


## 3. Contexto del caso de uso

Caso B necesita `t_outdoor`, `ghi`, `dewpoint`, `is_daylight`.


## 4. Relación con CENTINELA+

Features se calculan al vuelo en producción.


## 5. Relación con Medallion

Lee plata, escribe oro.


## 6. Datos de entrada

Mock ERA5.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos y derivamos.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "era5_xativa_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).set_index("timestamp")

def magnus_dewpoint(t, rh):
    a, b = 17.625, 243.04
    rh = np.clip(rh, 1, 100)
    alpha = np.log(rh / 100.0) + (a * t) / (b + t)
    return (b * alpha) / (a - alpha)

# RH no está en el mock; aproximamos con anti-correlación de T
rh_mock = np.clip(70 - (df["t_air_c"] - 18) * 1.5, 30, 90)
df["dewpoint_c"] = magnus_dewpoint(df["t_air_c"], rh_mock)
df["is_daylight"] = (df["ghi_w_m2"] > 50).astype(int)
df.head()


## 10. Exploración paso a paso

Estadísticas.


In [ ]:
print(df[["t_air_c", "ghi_w_m2", "dewpoint_c", "is_daylight"]].describe().round(2))


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Persistimos.


In [ ]:
out_dir = ROOT / "output" / "case_E"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "weather_features.parquet"
df.to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)}")


## 13. Visualizaciones explicativas

Dewpoint vs T_air.


In [ ]:
plt.figure(figsize=(7, 3))
plt.scatter(df["t_air_c"], df["dewpoint_c"], s=4, color="#3F51B5")
plt.plot([0, 35], [0, 35], "--", color="gray")
plt.xlabel("T air (°C)"); plt.ylabel("Dewpoint (°C)")
plt.title("Dewpoint sigue a T_air pero queda por debajo")
plt.tight_layout()


## 14. Validaciones

Dewpoint <= T_air siempre.


In [ ]:
assert (df["dewpoint_c"] <= df["t_air_c"] + 0.5).all()
print("Sanity check OK")


## 15. Errores comunes

1. Usar RH > 100 sin clip.
2. Aplicar Magnus a T en K.
3. Promediar `is_daylight` (categórica).


## 16. Ejercicios propuestos

1. Añade `solar_zenith_angle`.
2. Calcula `solar_irradiance_clear_sky` y compara.
3. Estima `wind_chill`.


## 17. Cómo se reutiliza con datos reales

Idéntico — solo cambia origen.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/04_prediccion_solar.ipynb`.
- Documento web del caso: `docs/use-cases/case-e-weather-solar.md`.
